## Table of Contents

* [Imported Libraries](#imported-libraries)
* [Hospital Data Access Example](#hospital-data-access-example)
  * [Using qualifed names only to distinguish duplicate intentional elements](#using-qualifed-names-only-to-distinguish-duplicate-intentional-elements)
  * [Using fully qualified names](#using-fully-qualified-names)
* [Order to Cash Example](#order-to-cash-example)
* [Equipment Rental Example](#equipment-rental-example)
* [Form Check Example](#form-check-example)
* [Incident Decision Second Trial Example](#incident-decision-second-trial-example)
* [Invoice Handled Example](#invoice-handled-example)

# Imported Libraries

In [ ]:
from pathlib import Path
from Semantics.event_mapping_from_csv import read_event_mapping_csv
from Semantics.istar_processor import read_istar_model
from Semantics.petri_net_processor import read_petri_net
from Semantics.bpmn_processor import read_from_bpmn
from Semantics.goccva_pipeline import analyse, create_report
from Semantics.goccva_helpers import sequences_to_event_log, create_event_log
from Ui.goccva_ui import show_goal_oriented_alignment

import pandas as pd
from IPython.display import display, Math


# Hospital Data Access Example

## Using qualifed names only to distinguish duplicate intentional elements

In [ ]:
goal_model_path = 'content/gm.txt'
process_model_path = 'content/pm.pnml'
mapping_path = 'content/mapping.csv'

goal_model = read_istar_model(goal_model_path)
petri_net = read_petri_net(process_model_path)
mapping = read_event_mapping_csv(mapping_path)

from pprint import pp

print("Goal model elements:")

pp(goal_model.elements())

targets = ["Data access provided", "(Hospital Officer) data easily accesible", "data easily accesible"]

# Generate the event log from the process model
# _ , event_log = create_event_log(petri_net, 100, 0.85)

# Build the event log by hand: one trace as an ordered list of activities.
activities = [
    "Identity verified",
    "Provide Records",
    "Format Data Regular Needs",
    "Format Data Special Needs",
]
event_log = sequences_to_event_log([activities])

summary, detailed, contribution_to_targets = analyse(goal_model, petri_net, event_log, targets, mapping)

trace_id = 1

print("Summary")
summary_df = pd.DataFrame(summary)
display(summary_df)

print(f"\nDetailed (trace_id = {trace_id}, goal-oriented alignment)")
show_goal_oriented_alignment(summary, detailed, trace_id)

print("\nContribution to targets")
# targets_df = pd.DataFrame(contribution_to_targets)
# display(targets_df)
pp(contribution_to_targets)

# report = create_report(summary)
# print(report)

## Using fully qualified names

In [ ]:
goal_model_path = 'content/gm.txt'
process_model_path = 'content/pm.pnml'
mapping_path = 'content/mapping_qualified.csv'

goal_model = read_istar_model(goal_model_path,qualified=True)
petri_net = read_petri_net(process_model_path)
mapping = read_event_mapping_csv(mapping_path)

from pprint import pp

print("Goal model elements:")
pp(goal_model.elements())

targets = ["(Hospital Officer) Data access provided", "(Hospital Officer) data easily accesible", "data easily accesible"]

# Generate the event log from the process model
# _ , event_log = create_event_log(petri_net, 100, 0.85)

# Build the event log by hand: one trace as an ordered list of activities.
activities = [
    "Identity verified",
    "Provide Records",
    "Format Data Regular Needs",
    "Format Data Special Needs",
]
event_log = sequences_to_event_log([activities])

summary, detailed, contribution_to_targets = analyse(goal_model, petri_net, event_log, targets, mapping)

trace_id = 1

print("Summary")
summary_df = pd.DataFrame(summary)
display(summary_df)

print(f"\nDetailed (trace_id = {trace_id}, goal-oriented alignment)")
show_goal_oriented_alignment(summary, detailed, trace_id)

print("\nContribution to targets")
# targets_df = pd.DataFrame(contribution_to_targets)
# display(targets_df)
pp(contribution_to_targets)

# report = create_report(summary)
# print(report)

# Order to Cash Example

In [ ]:
goal_model_path = 'content/case2/ordertocashgm.txt'
process_model_path = 'content/case2/ordertocashpm.bpmn'

goal_model = read_istar_model(goal_model_path)

# from pprint import pp

petri_net, mapping = read_from_bpmn(process_model_path)

targets = ['Product shipped', 'Order fulfilled ', 'Order confirmed']

# Generate the event log from the process model
_ , event_log = create_event_log(petri_net, 100, 0.85)

# pp(goal_model.leaves())
# Build the event log by hand: one trace as an ordered list of activities.
# activities = [
#  'Check stock availability',
#  'Retrieve product from warehouse',
#  'Confirm order',
#  'Get shipment address',
#  'Ship product',
#  'Emit invoice',
#  'Receive payment',
#  'Archive order',
#  ]
# event_log = sequences_to_event_log([activities])

summary, detailed, contribution_to_targets = analyse(goal_model, petri_net, event_log, targets, mapping)

print("Summary")
summary_df = pd.DataFrame(summary)
display(summary_df)

non_optimal = summary_df[summary_df['traditional_class'] == 'non-optimal']
candidates = non_optimal if not non_optimal.empty else summary_df
trace_id = candidates.loc[candidates['trace'].apply(len).idxmax(), 'trace_id']

print(f"\nDetailed (trace_id = {trace_id}, goal-oriented alignment)")
show_goal_oriented_alignment(summary, detailed, trace_id)

print("\nContribution to targets")
targets_df = pd.DataFrame(contribution_to_targets)
display(targets_df)

report = create_report(summary)
print(report)

# Equipment Rental Example

In [ ]:
goal_model_path = 'content/case3/equipmentrentalgm.txt'
process_model_path = 'content/case3/equipmentrentalpm.bpmn'

goal_model = read_istar_model(goal_model_path)

from pprint import pp

# print("Leaves:")
# pp(goal_model.leaves())

# print("Goals and Qualities:")
# pp(goal_model.elements() - goal_model.leaves())

petri_net, mapping = read_from_bpmn(process_model_path)

# pp(mapping)

targets = ['PO Created']

# Generate the event log from the process model
# _ , event_log = create_event_log(petri_net, 10, 0.85)

# pp(goal_model.leaves())
# Build the event log by hand: one trace as an ordered list of activities.
activities = [
 'Submit equipment rental request',
 'Request equipment for rent',
 'Select suitable equipment',
 'Check availability',
 'equipment available',
 'Review rental request',
 'Approve Rental request',
 'Create PO',
]
event_log = sequences_to_event_log([activities])

summary, detailed, contribution_to_targets = analyse(goal_model, petri_net, event_log, targets, mapping)

print("Summary")
summary_df = pd.DataFrame(summary)
display(summary_df)

non_optimal = summary_df[summary_df['traditional_class'] == 'non-optimal']
candidates = non_optimal if not non_optimal.empty else summary_df
trace_id = candidates.loc[candidates['trace'].apply(len).idxmax(), 'trace_id']

print(f"\nDetailed (trace_id = {trace_id}, goal-oriented alignment)")
show_goal_oriented_alignment(summary, detailed, trace_id)

print("\nContribution to targets")
targets_df = pd.DataFrame(contribution_to_targets)
display(targets_df)

report = create_report(summary)
print(report)

# Form Check Example

In [ ]:
goal_model_path = 'content/case4/formcheckedgm.txt'
process_model_path = 'content/case4/formcheckedpm.bpmn'

goal_model = read_istar_model(goal_model_path)

# from pprint import pp

# print("Leaves:")
# pp(goal_model.leaves())

# print("Goals and Qualities:")
# pp(goal_model.elements() - goal_model.leaves())

petri_net, mapping = read_from_bpmn(process_model_path)

# pp(mapping)

targets = [
 'form complete',
 ]

# Generate the event log from the process model
_ , event_log = create_event_log(petri_net, 100, 0.85)

# Build the event log by hand: one trace as an ordered list of activities.
# activities = [
#     'Check application form completeness',
#     'Return application back to applicant',
#     'Receive updated application',
#     'Check application form completeness',
# ]
# event_log = sequences_to_event_log([activities])

summary, detailed, contribution_to_targets = analyse(goal_model, petri_net, event_log, targets, mapping)

print("Summary")
summary_df = pd.DataFrame(summary)
display(summary_df)

non_optimal = summary_df[summary_df['traditional_class'] == 'non-optimal']
candidates = non_optimal if not non_optimal.empty else summary_df
trace_id = candidates.loc[candidates['trace'].apply(len).idxmax(), 'trace_id']

print(f"\nDetailed (trace_id = {trace_id}, goal-oriented alignment)")
show_goal_oriented_alignment(summary, detailed, trace_id)

print("\nContribution to targets")
targets_df = pd.DataFrame(contribution_to_targets)
display(targets_df)

report = create_report(summary)
print(report)

# Incident Decision Second Trial Example

In [ ]:
goal_model_path = 'content/case5/incdecisionsecondtrialgm.txt'
process_model_path = 'content/case5/incdecisionsecondtrialpm.bpmn'

goal_model = read_istar_model(goal_model_path)

from pprint import pp

# print("Leaves:")
# pp(goal_model.leaves())

# print("Goals and Qualities:")
# pp(goal_model.elements() - goal_model.leaves())

petri_net, mapping = read_from_bpmn(process_model_path)

# pp(petri_net.net.transitions)

# pp(mapping)

targets = [
 'Order completed',
 ]

# Generate the event log from the process model
_ , event_log = create_event_log(petri_net, 1000, 0.85)

# Build the event log by hand: one trace as an ordered list of activities.
# activities = [
#     'Check application form completeness',
#     'Return application back to applicant',
#     'Receive updated application',
#     'Check application form completeness',
# ]
# event_log = sequences_to_event_log([activities])

summary, detailed, contribution_to_targets = analyse(goal_model, petri_net, event_log, targets, mapping)

print("Summary")
summary_df = pd.DataFrame(summary)
display(summary_df)

non_optimal = summary_df[summary_df['traditional_class'] == 'non-optimal']
candidates = non_optimal if not non_optimal.empty else summary_df
trace_id = candidates.loc[candidates['trace'].apply(len).idxmax(), 'trace_id']

print(f"\nDetailed (trace_id = {trace_id}, goal-oriented alignment)")
show_goal_oriented_alignment(summary, detailed, trace_id)

print("\nContribution to targets")
targets_df = pd.DataFrame(contribution_to_targets)
display(targets_df)

report = create_report(summary)
print(report)

# Invoice Handled Example

In [ ]:
goal_model_path = 'content/case6/invoicehandledgm.txt'
process_model_path = 'content/case6/invoicehandledpm.bpmn'

goal_model = read_istar_model(goal_model_path)

# from pprint import pp

# print("Leaves:")
# pp(goal_model.leaves())

# print("Goals and Qualities:")
# pp(goal_model.elements() - goal_model.leaves())

petri_net, mapping = read_from_bpmn(process_model_path)

# pp(petri_net.net.transitions)

# pp(mapping)

targets = [
 'Invoice Handled',
 'zero mistmatches'
 ]

# Generate the event log from the process model
_ , event_log = create_event_log(petri_net, 1000, 0.85)

# Build the event log by hand: one trace as an ordered list of activities.
# activities = [
#     'Check application form completeness',
#     'Return application back to applicant',
#     'Receive updated application',
#     'Check application form completeness',
# ]
# event_log = sequences_to_event_log([activities])

summary, detailed, contribution_to_targets = analyse(goal_model, petri_net, event_log, targets, mapping)

print("Summary")
summary_df = pd.DataFrame(summary)
display(summary_df)

non_optimal = summary_df[summary_df['traditional_class'] == 'non-optimal']
candidates = non_optimal if not non_optimal.empty else summary_df
trace_id = candidates.loc[candidates['trace'].apply(len).idxmax(), 'trace_id']

print(f"\nDetailed (trace_id = {trace_id}, goal-oriented alignment)")
show_goal_oriented_alignment(summary, detailed, trace_id)

print("\nContribution to targets")
targets_df = pd.DataFrame(contribution_to_targets)
display(targets_df)

report = create_report(summary)
print(report)